In [ ]:
# ====================== 0. Multi-stock robustness framework ======================
print("="*80)
print("多股票稳健性测试：TCN、DLinear、LSTM模型对比")
print("="*80)
print("导入所需库...")

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                           f1_score, confusion_matrix, roc_auc_score, 
                           average_precision_score, roc_curve, precision_recall_curve)
from torch.utils.data import TensorDataset, DataLoader
import os
import time
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from scipy.stats import pearsonr
import json
from tqdm import tqdm

# Configure fonts for Chinese display text.
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
mpl.rcParams['font.family'] = 'sans-serif'

# Set the random seed.
torch.manual_seed(42)
np.random.seed(42)

print("✓ 库导入和设置完成")

# Global parameters
SEQ_LEN = 60
PRED_DAYS = 5
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
BATCH_SIZE = 32
EPOCHS = 50
LR = 0.001
PATIENCE = 10

# Select stocks to evaluate.
STOCK_SYMBOLS = ['AAPL', 'MSFT', 'JPM', 'JNJ', 'WMT']  # Five stocks from different sectors
# STOCK_SYMBOLS = ['AAPL', 'MSFT', 'GOOGL']  # Use these stocks first if data are limited.

print(f"将测试以下股票: {STOCK_SYMBOLS}")
print(f"全局参数: 序列长度={SEQ_LEN}, 预测窗口={PRED_DAYS}天")

In [ ]:
# ====================== 1. Utility functions ======================
print("\n" + "="*80)
print("步骤1: 工具函数定义")
print("="*80)

# Technical-indicator functions, unchanged from the earlier implementation
def compute_rsi(series, window=14):
    delta = series.diff()
    gain = delta.copy()
    loss = delta.copy()
    gain[gain < 0] = 0
    loss[loss > 0] = 0
    loss = loss.abs()
    avg_gain = gain.rolling(window).mean()
    avg_loss = loss.rolling(window).mean()
    avg_loss = avg_loss.replace(0, 1e-10)
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

def compute_atr(high, low, close, window=14):
    tr1 = high - low
    tr2 = abs(high - close.shift())
    tr3 = abs(low - close.shift())
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    return tr.rolling(window).mean()

def compute_stochastic(high, low, close, k_window=14, d_window=3):
    lowest_low = low.rolling(k_window).min()
    highest_high = high.rolling(k_window).max()
    stoch_k = 100 * (close - lowest_low) / (highest_high - lowest_low)
    stoch_d = stoch_k.rolling(d_window).mean()
    return stoch_k, stoch_d

def compute_volume_indicators(volume, close, window=20):
    volume_ma = volume.rolling(window).mean()
    volume_ratio = volume / volume_ma
    obv = (np.sign(close.diff()) * volume).fillna(0).cumsum()
    return volume_ma, volume_ratio, obv

def compute_macd(series, fast_period=12, slow_period=26, signal_period=9):
    ema_fast = series.ewm(span=fast_period, adjust=False).mean()
    ema_slow = series.ewm(span=slow_period, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal_period, adjust=False).mean()
    histogram = macd_line - signal_line
    return macd_line, signal_line, histogram

# Data-loading function
def load_stock_data(symbol, data_dir='./Dow 30 Daily', start_date='2000-01-01', end_date='2019-12-31'):
    """Load data for one stock."""
    file_path = os.path.join(data_dir, f"{symbol}.txt")
    
    if not os.path.exists(file_path):
        print(f"⚠️ 文件不存在: {file_path}")
        return None
    
    print(f"加载 {symbol} 数据...")
    df = pd.read_csv(file_path, header=None, 
                     names=['Date', 'Open', 'High', 'Low', 'Close', 'Volume'])
    df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y')
    df.set_index('Date', inplace=True)
    
    # Filter the date range.
    mask = (df.index >= start_date) & (df.index <= end_date)
    df_filtered = df.loc[mask].copy()
    
    if len(df_filtered) == 0:
        print(f"⚠️ {symbol} 在指定时间范围内无数据")
        return None
    
    # Calculate log returns.
    df_filtered['LogReturn'] = np.log(df_filtered['Close']).diff()
    
    # Calculate technical indicators.
    df_filtered['RSI'] = compute_rsi(df_filtered['Close'], window=14)
    df_filtered['Stoch_K'], df_filtered['Stoch_D'] = compute_stochastic(
        df_filtered['High'], df_filtered['Low'], df_filtered['Close']
    )
    df_filtered['ATR'] = compute_atr(df_filtered['High'], df_filtered['Low'], df_filtered['Close'], window=14)
    df_filtered['MACD'], df_filtered['MACD_Signal'], df_filtered['MACD_Histogram'] = compute_macd(df_filtered['Close'])
    df_filtered['Volume_MA'], df_filtered['Volume_Ratio'], df_filtered['OBV'] = compute_volume_indicators(
        df_filtered['Volume'], df_filtered['Close'], window=20
    )
    
    # Remove raw prices and volume.
    df_technical = df_filtered.drop(['Open', 'High', 'Low', 'Close', 'Volume'], axis=1)
    df_technical.dropna(inplace=True)
    
    print(f"  {symbol}: {len(df_technical)} 样本, 时间范围: {df_technical.index.min().date()} 到 {df_technical.index.max().date()}")
    return df_technical

# Sequence-construction function
def create_sequences(data, seq_len=60, pred_days=5):
    """Construct sequence data."""
    feature_columns = [col for col in data.columns if col != 'LogReturn']
    X, y = [], []
    
    features = data[feature_columns].values
    returns = data['LogReturn'].values
    
    for i in range(seq_len, len(data) - pred_days):
        input_seq = features[i-seq_len:i]
        future_returns = returns[i:i+pred_days]
        avg_return = np.mean(future_returns)
        label = 1 if avg_return >= 0 else 0
        
        X.append(input_seq)
        y.append(label)
    
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

# Data-standardization function
def standardize_data(train_features, val_features, test_features):
    """Standardize data."""
    scaler = StandardScaler()
    train_features_scaled = scaler.fit_transform(train_features)
    val_features_scaled = scaler.transform(val_features)
    test_features_scaled = scaler.transform(test_features)
    return train_features_scaled, val_features_scaled, test_features_scaled, scaler

print("✓ 工具函数定义完成")

In [ ]:
# ====================== 2. Model definitions ======================
print("\n" + "="*80)
print("步骤2: 模型定义")
print("="*80)

# TCN components
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super(Chomp1d, self).__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout=0.2):
        super(TemporalBlock, self).__init__()
        self.conv1 = nn.utils.weight_norm(nn.Conv1d(n_inputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation))
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.conv2 = nn.utils.weight_norm(nn.Conv1d(n_outputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation))
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1,
                                 self.conv2, self.chomp2, self.relu2, self.dropout2)
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TemporalConvNet(nn.Module):
    def __init__(self, num_inputs, num_channels, kernel_size=2, dropout=0.2):
        super(TemporalConvNet, self).__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = num_inputs if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers += [TemporalBlock(in_channels, out_channels, kernel_size, stride=1, 
                                     dilation=dilation_size, padding=(kernel_size-1) * dilation_size, dropout=dropout)]
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        return self.network(x)

class TCNClassifier(nn.Module):
    def __init__(self, input_size, seq_len, num_channels=[32, 32, 32], kernel_size=3, dropout=0.2):
        super(TCNClassifier, self).__init__()
        self.tcn = TemporalConvNet(input_size, num_channels, kernel_size, dropout)
        self.linear = nn.Linear(num_channels[-1], 1)
    def forward(self, x):
        x = x.transpose(1, 2)
        y = self.tcn(x)
        output = self.linear(y[:, :, -1])
        return output

# DLinear components
class moving_avg(nn.Module):
    def __init__(self, kernel_size, stride):
        super(moving_avg, self).__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=stride, padding=0)
    def forward(self, x):
        front = x[:, 0:1, :].repeat(1, (self.kernel_size - 1) // 2, 1)
        end = x[:, -1:, :].repeat(1, (self.kernel_size - 1) // 2, 1)
        x = torch.cat([front, x, end], dim=1)
        x = self.avg(x.permute(0, 2, 1))
        x = x.permute(0, 2, 1)
        return x

class series_decomp(nn.Module):
    def __init__(self, kernel_size):
        super(series_decomp, self).__init__()
        self.moving_avg = moving_avg(kernel_size, stride=1)
    def forward(self, x):
        moving_mean = self.moving_avg(x)
        res = x - moving_mean
        return res, moving_mean

class DLinearClassifier(nn.Module):
    def __init__(self, seq_len, num_features, pred_len=5, kernel_size=25, dropout=0.2):
        super(DLinearClassifier, self).__init__()
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.num_features = num_features
        self.decomposition = series_decomp(kernel_size)
        self.linear_trend = nn.Linear(seq_len, pred_len)
        self.linear_seasonal = nn.Linear(seq_len, pred_len)
        self.classifier = nn.Sequential(
            nn.Linear(num_features * pred_len, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        x = x.transpose(1, 2)
        seasonal_init, trend_init = self.decomposition(x)
        trend_output = self.linear_trend(trend_init)
        seasonal_output = self.linear_seasonal(seasonal_init)
        output = trend_output + seasonal_output
        output = output.transpose(1, 2)
        output = output.reshape(output.size(0), -1)
        output = self.classifier(output)
        return output

# LSTM model
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super(LSTMClassifier, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                          num_layers=num_layers, batch_first=True,
                          dropout=dropout if num_layers > 1 else 0, bidirectional=False)
        self.dropout = nn.Dropout(dropout)
        self.linear = nn.Linear(hidden_size, 1)
    def forward(self, x):
        lstm_out, (hidden, cell) = self.lstm(x)
        lstm_out = self.dropout(lstm_out[:, -1, :])
        output = self.linear(lstm_out)
        return output

print("✓ 模型定义完成")

In [ ]:
# ====================== 3. Training and evaluation functions ======================
print("\n" + "="*80)
print("步骤3: 训练和评估函数")
print("="*80)

def calculate_metrics(predictions, targets, threshold=0.5):
    """Calculate evaluation metrics."""
    pred_binary = (predictions >= threshold).astype(int)
    
    accuracy = accuracy_score(targets, pred_binary)
    precision = precision_score(targets, pred_binary, zero_division=0)
    recall = recall_score(targets, pred_binary, zero_division=0)
    f1 = f1_score(targets, pred_binary, zero_division=0)
    
    if len(np.unique(targets)) > 1:
        roc_auc = roc_auc_score(targets, predictions)
        pr_auc = average_precision_score(targets, predictions)
    else:
        roc_auc = 0.5
        pr_auc = 0.5
    
    if len(np.unique(predictions)) > 1 and len(np.unique(targets)) > 1:
        ic, ic_pvalue = pearsonr(predictions, targets)
    else:
        ic, ic_pvalue = 0.0, 1.0
    
    cm = confusion_matrix(targets, pred_binary)
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
    
    return {
        'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1,
        'roc_auc': roc_auc, 'pr_auc': pr_auc, 'ic': ic, 'ic_pvalue': ic_pvalue,
        'confusion_matrix': cm, 'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn
    }

def find_optimal_threshold(predictions, targets, metric='f1'):
    """Select the optimal threshold on the validation set."""
    thresholds = np.linspace(0.1, 0.9, 81)
    best_threshold = 0.5
    best_score = 0
    
    for threshold in thresholds:
        pred_binary = (predictions >= threshold).astype(int)
        if metric == 'f1':
            score = f1_score(targets, pred_binary, zero_division=0)
        elif metric == 'accuracy':
            score = accuracy_score(targets, pred_binary)
        else:
            continue
        
        if score > best_score:
            best_score = score
            best_threshold = threshold
    
    return best_threshold, best_score

def train_model(model, model_name, train_loader, val_loader, num_epochs=50, lr=0.001, patience=10):
    """Train one model."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    
    best_val_f1 = 0
    epochs_no_improve = 0
    best_model_state = None
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss, train_preds, train_targets = 0, [], []
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch).squeeze()
            loss = criterion(outputs, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
            with torch.no_grad():
                probs = torch.sigmoid(outputs).cpu().numpy()
                train_preds.extend(probs)
                train_targets.extend(y_batch.cpu().numpy())
        
        # Validation phase
        model.eval()
        val_loss, val_preds, val_targets = 0, [], []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch).squeeze()
                loss = criterion(outputs, y_batch)
                val_loss += loss.item()
                probs = torch.sigmoid(outputs).cpu().numpy()
                val_preds.extend(probs)
                val_targets.extend(y_batch.cpu().numpy())
        
        # Calculate metrics.
        val_metrics = calculate_metrics(np.array(val_preds), np.array(val_targets))
        current_val_f1 = val_metrics['f1']
        
        # Schedule the learning rate.
        scheduler.step(val_loss / len(val_loader))
        
        # Early stopping
        if current_val_f1 > best_val_f1:
            best_val_f1 = current_val_f1
            epochs_no_improve = 0
            best_model_state = model.state_dict().copy()
        else:
            epochs_no_improve += 1
        
        if epochs_no_improve >= patience:
            break
    
    # Load the best model.
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    return model, best_val_f1

def evaluate_model(model, data_loader, threshold=0.5):
    """Evaluate a model."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    model.to(device)
    
    all_probs, all_targets = [], []
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch).squeeze()
            probs = torch.sigmoid(outputs).cpu().numpy()
            all_probs.extend(probs)
            all_targets.extend(y_batch.cpu().numpy())
    
    return np.array(all_probs), np.array(all_targets)

print("✓ 训练和评估函数定义完成")

In [ ]:
# ====================== 4. Main experiment ======================
print("\n" + "="*80)
print("步骤4: 多股票实验主函数")
print("="*80)

def run_stock_experiment(symbol, data_dir='./Dow 30 Daily'):
    """Run the complete experiment for one stock."""
    print(f"\n{'='*60}")
    print(f"处理股票: {symbol}")
    print(f"{'='*60}")
    
    # 1. Load data.
    df = load_stock_data(symbol, data_dir)
    if df is None or len(df) < 1000:
        print(f"⚠️  {symbol} 数据不足，跳过")
        return None
    
    # 2. Split the dataset.
    total_size = len(df)
    train_size = int(total_size * TRAIN_RATIO)
    val_size = int(total_size * VAL_RATIO)
    test_size = total_size - train_size - val_size
    
    train_df = df.iloc[:train_size]
    val_df = df.iloc[train_size:train_size+val_size]
    test_df = df.iloc[train_size+val_size:]
    
    print(f"  数据集划分: 训练{len(train_df)} 验证{len(val_df)} 测试{len(test_df)}")
    print(f"  时间范围: 训练[{train_df.index.min().date()} 到 {train_df.index.max().date()}]")
    print(f"            测试[{test_df.index.min().date()} 到 {test_df.index.max().date()}]")
    
    # 3. Construct sequences.
    X_train_raw, y_train = create_sequences(train_df, SEQ_LEN, PRED_DAYS)
    X_val_raw, y_val = create_sequences(val_df, SEQ_LEN, PRED_DAYS)
    X_test_raw, y_test = create_sequences(test_df, SEQ_LEN, PRED_DAYS)
    
    if len(X_train_raw) == 0 or len(X_val_raw) == 0 or len(X_test_raw) == 0:
        print(f"⚠️  {symbol} 序列创建失败，跳过")
        return None
    
    # 4. Standardize.
    train_samples, train_timesteps, train_features = X_train_raw.shape
    val_samples, val_timesteps, val_features = X_val_raw.shape
    test_samples, test_timesteps, test_features = X_test_raw.shape
    
    X_train_flat = X_train_raw.reshape(-1, train_features)
    X_val_flat = X_val_raw.reshape(-1, train_features)
    X_test_flat = X_test_raw.reshape(-1, train_features)
    
    X_train_flat_scaled, X_val_flat_scaled, X_test_flat_scaled, _ = standardize_data(
        X_train_flat, X_val_flat, X_test_flat
    )
    
    X_train = X_train_flat_scaled.reshape(train_samples, train_timesteps, train_features)
    X_val = X_val_flat_scaled.reshape(val_samples, val_timesteps, train_features)
    X_test = X_test_flat_scaled.reshape(test_samples, test_timesteps, train_features)
    
    # Label distribution
    pos_ratio_train = np.mean(y_train)
    pos_ratio_val = np.mean(y_val)
    pos_ratio_test = np.mean(y_test)
    print(f"  正样本比例: 训练{pos_ratio_train:.3f} 验证{pos_ratio_val:.3f} 测试{pos_ratio_test:.3f}")
    
    # 5. Prepare data loaders.
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32)
    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val, dtype=torch.float32)
    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    y_test_t = torch.tensor(y_test, dtype=torch.float32)
    
    train_dataset = TensorDataset(X_train_t, y_train_t)
    val_dataset = TensorDataset(X_val_t, y_val_t)
    test_dataset = TensorDataset(X_test_t, y_test_t)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    # 6. Initialize models.
    num_features = X_train.shape[2]
    
    tcn_model = TCNClassifier(input_size=num_features, seq_len=SEQ_LEN, 
                              num_channels=[32, 32, 32], kernel_size=3, dropout=0.2)
    dlinear_model = DLinearClassifier(seq_len=SEQ_LEN, num_features=num_features, 
                                      pred_len=PRED_DAYS, kernel_size=25, dropout=0.2)
    lstm_model = LSTMClassifier(input_size=num_features, hidden_size=64, 
                                num_layers=2, dropout=0.2)
    
    models = {
        'TCN': tcn_model,
        'DLinear': dlinear_model,
        'LSTM': lstm_model
    }
    
    # 7. Train and evaluate.
    results = {}
    
    for model_name, model in models.items():
        print(f"\n  训练 {model_name}...")
        
        # Train the model.
        trained_model, best_val_f1 = train_model(
            model, model_name, train_loader, val_loader, 
            num_epochs=EPOCHS, lr=LR, patience=PATIENCE
        )
        
        # Get validation predictions and select the optimal threshold.
        val_probs, val_targets = evaluate_model(trained_model, val_loader)
        best_threshold, best_f1 = find_optimal_threshold(val_probs, val_targets, metric='f1')
        
        # Get test predictions.
        test_probs, test_targets = evaluate_model(trained_model, test_loader)
        
        # Calculate test metrics.
        test_metrics = calculate_metrics(test_probs, test_targets, threshold=best_threshold)
        
        # Save results.
        results[model_name] = {
            'best_threshold': best_threshold,
            'best_val_f1': best_val_f1,
            'test_metrics': test_metrics,
            'pos_ratio': {'train': pos_ratio_train, 'val': pos_ratio_val, 'test': pos_ratio_test},
            'sample_size': {'train': len(X_train), 'val': len(X_val), 'test': len(X_test)}
        }
        
        print(f"    最优阈值: {best_threshold:.3f}, 验证F1: {best_val_f1:.3f}")
        print(f"    测试F1: {test_metrics['f1']:.3f}, ROC-AUC: {test_metrics['roc_auc']:.3f}, IC: {test_metrics['ic']:.3f}")
    
    return {
        'symbol': symbol,
        'results': results,
        'data_info': {
            'time_range': {
                'train': f"{train_df.index.min().date()} 到 {train_df.index.max().date()}",
                'test': f"{test_df.index.min().date()} 到 {test_df.index.max().date()}"
            },
            'total_samples': len(df)
        }
    }

print("✓ 实验主函数定义完成")

In [ ]:
# ====================== 5. Run the multi-stock experiment ======================
print("\n" + "="*80)
print("步骤5: 运行多股票稳健性测试")
print("="*80)

# Store results for all stocks.
all_results = {}

# Iterate over all stocks.
for symbol in tqdm(STOCK_SYMBOLS, desc="处理股票"):
    try:
        result = run_stock_experiment(symbol)
        if result is not None:
            all_results[symbol] = result
    except Exception as e:
        print(f"❌ 处理 {symbol} 时出错: {e}")
        continue

# Save results.
if all_results:
    with open('multi_stock_results.json', 'w') as f:
        json.dump(all_results, f, indent=2, default=str)
    print(f"\n✓ 结果已保存到: multi_stock_results.json")
else:
    print("❌ 没有成功处理的股票")

In [ ]:
# ====================== 6. Summarize and analyze results ======================
print("\n" + "="*80)
print("步骤6: 结果汇总和分析")
print("="*80)

if not all_results:
    print("没有可用的结果进行分析")
else:
    # Create a summary table.
    summary_data = []
    
    for symbol, data in all_results.items():
        for model_name, model_data in data['results'].items():
            test_metrics = model_data['test_metrics']
            summary_data.append({
                'Stock': symbol,
                'Model': model_name,
                'Accuracy': test_metrics['accuracy'],
                'Precision': test_metrics['precision'],
                'Recall': test_metrics['recall'],
                'F1': test_metrics['f1'],
                'ROC-AUC': test_metrics['roc_auc'],
                'PR-AUC': test_metrics['pr_auc'],
                'IC': test_metrics['ic'],
                'IC_pvalue': test_metrics['ic_pvalue'],
                'Threshold': model_data['best_threshold'],
                'Val_F1': model_data['best_val_f1'],
                'Train_Samples': model_data['sample_size']['train'],
                'Test_Samples': model_data['sample_size']['test'],
                'Pos_Ratio_Test': model_data['pos_ratio']['test']
            })
    
    summary_df = pd.DataFrame(summary_data)
    
    # Save the summary table.
    summary_df.to_csv('multi_stock_summary.csv', index=False)
    
    print("汇总结果表格:")
    print("="*80)
    print(summary_df[['Stock', 'Model', 'F1', 'ROC-AUC', 'IC', 'IC_pvalue', 'Threshold']].to_string())
    
    # Statistical analysis
    print("\n统计分析:")
    print("="*80)
    
    # Summarize mean performance by model.
    model_stats = summary_df.groupby('Model').agg({
        'F1': ['mean', 'std', 'min', 'max'],
        'ROC-AUC': ['mean', 'std'],
        'IC': ['mean', 'std']
    }).round(4)
    
    print("各模型平均性能:")
    print(model_stats)
    
    # Find the best model for each stock.
    best_models = []
    for symbol in all_results.keys():
        symbol_data = summary_df[summary_df['Stock'] == symbol]
        best_f1_idx = symbol_data['F1'].idxmax()
        best_model = symbol_data.loc[best_f1_idx]
        best_models.append({
            'Stock': symbol,
            'Best_Model': best_model['Model'],
            'Best_F1': best_model['F1'],
            'Best_ROC_AUC': best_model['ROC-AUC'],
            'Best_IC': best_model['IC']
        })
    
    best_models_df = pd.DataFrame(best_models)
    print(f"\n每只股票的最佳模型:")
    print(best_models_df.to_string(index=False))
    
    # Calculate DLinear performance.
    dlinear_results = summary_df[summary_df['Model'] == 'DLinear']
    print(f"\nDLinear模型表现统计 ({len(dlinear_results)}只股票):")
    print(f"  F1平均值: {dlinear_results['F1'].mean():.4f} ± {dlinear_results['F1'].std():.4f}")
    print(f"  ROC-AUC平均值: {dlinear_results['ROC-AUC'].mean():.4f} ± {dlinear_results['ROC-AUC'].std():.4f}")
    print(f"  IC平均值: {dlinear_results['IC'].mean():.4f} ± {dlinear_results['IC'].std():.4f}")
    
    # Count stocks with significantly positive correlation.
    significant_ic = summary_df[(summary_df['IC_pvalue'] < 0.05) & (summary_df['IC'] > 0)]
    print(f"\nIC显著为正的模型数量: {len(significant_ic)}/{len(summary_df)}")
    for model in ['TCN', 'DLinear', 'LSTM']:
        model_sig = significant_ic[significant_ic['Model'] == model]
        print(f"  {model}: {len(model_sig)}/{len(summary_df[summary_df['Model']==model])}")

In [ ]:
# ====================== 7. Visualize results ======================
print("\n" + "="*80)
print("步骤7: 可视化结果")
print("="*80)

if not all_results:
    print("没有可用的结果进行可视化")
else:
    # Create visualizations.
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # 1. Compare model F1 scores.
    models = ['TCN', 'DLinear', 'LSTM']
    colors = ['skyblue', 'lightcoral', 'lightgreen']
    
    for i, model in enumerate(models):
        model_data = summary_df[summary_df['Model'] == model]
        axes[0, 0].bar(np.arange(len(model_data)) + i*0.3, model_data['F1'], 
                      width=0.3, color=colors[i], alpha=0.7, label=model)
    axes[0, 0].set_title('各股票F1分数对比')
    axes[0, 0].set_xlabel('股票')
    axes[0, 0].set_ylabel('F1分数')
    stocks = list(all_results.keys())
    axes[0, 0].set_xticks(np.arange(len(stocks)) + 0.3)
    axes[0, 0].set_xticklabels(stocks, rotation=45)
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3, axis='y')
    
    # 2. Compare mean model performance.
    avg_f1 = summary_df.groupby('Model')['F1'].mean()
    avg_roc = summary_df.groupby('Model')['ROC-AUC'].mean()
    
    x = np.arange(len(models))
    width = 0.35
    axes[0, 1].bar(x - width/2, avg_f1.values, width, label='平均F1', alpha=0.7)
    axes[0, 1].bar(x + width/2, avg_roc.values, width, label='平均ROC-AUC', alpha=0.7)
    axes[0, 1].set_title('各模型平均性能')
    axes[0, 1].set_xlabel('模型')
    axes[0, 1].set_ylabel('分数')
    axes[0, 1].set_xticks(x)
    axes[0, 1].set_xticklabels(models)
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3, axis='y')
    
    # 3. Boxplot of IC distributions
    ic_data = [summary_df[summary_df['Model']==model]['IC'].values for model in models]
    axes[0, 2].boxplot(ic_data, labels=models)
    axes[0, 2].set_title('信息系数(IC)分布')
    axes[0, 2].set_ylabel('IC')
    axes[0, 2].axhline(y=0, color='r', linestyle='--', alpha=0.5, label='IC=0')
    axes[0, 2].grid(True, alpha=0.3, axis='y')
    
    # 4. Best-model counts
    best_model_counts = best_models_df['Best_Model'].value_counts()
    axes[1, 0].pie(best_model_counts.values, labels=best_model_counts.index, 
                  autopct='%1.1f%%', colors=colors[:len(best_model_counts)])
    axes[1, 0].set_title('最佳模型分布')
    
    # 5. Positive-class fraction in the test set
    test_pos_ratios = []
    for symbol in stocks:
        if symbol in all_results:
            # Get the positive-class fraction from the first model.
            first_model = list(all_results[symbol]['results'].keys())[0]
            pos_ratio = all_results[symbol]['results'][first_model]['pos_ratio']['test']
            test_pos_ratios.append(pos_ratio)
    
    axes[1, 1].bar(range(len(stocks)), test_pos_ratios, alpha=0.7)
    axes[1, 1].axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='平衡点(0.5)')
    axes[1, 1].set_title('各股票测试集正样本比例')
    axes[1, 1].set_xlabel('股票')
    axes[1, 1].set_ylabel('正样本比例')
    axes[1, 1].set_xticks(range(len(stocks)))
    axes[1, 1].set_xticklabels(stocks, rotation=45)
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3, axis='y')
    
    # 6. Sample counts
    sample_sizes = []
    for symbol in stocks:
        if symbol in all_results:
            first_model = list(all_results[symbol]['results'].keys())[0]
            train_size = all_results[symbol]['results'][first_model]['sample_size']['train']
            test_size = all_results[symbol]['results'][first_model]['sample_size']['test']
            sample_sizes.append((train_size, test_size))
    
    train_sizes = [s[0] for s in sample_sizes]
    test_sizes = [s[1] for s in sample_sizes]
    
    x = np.arange(len(stocks))
    width = 0.35
    axes[1, 2].bar(x - width/2, train_sizes, width, label='训练集', alpha=0.7)
    axes[1, 2].bar(x + width/2, test_sizes, width, label='测试集', alpha=0.7)
    axes[1, 2].set_title('各股票样本数量')
    axes[1, 2].set_xlabel('股票')
    axes[1, 2].set_ylabel('样本数量')
    axes[1, 2].set_xticks(x)
    axes[1, 2].set_xticklabels(stocks, rotation=45)
    axes[1, 2].legend()
    axes[1, 2].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('multi_stock_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ 可视化图表已保存为: multi_stock_analysis.png")

In [ ]:
# ====================== 8. Confusion-matrix analysis ======================
print("\n" + "="*80)
print("步骤8: 混淆矩阵详细分析")
print("="*80)

if all_results:
    print("\n各模型在各股票上的混淆矩阵:")
    print("="*80)
    
    for symbol, data in all_results.items():
        print(f"\n{'='*60}")
        print(f"股票: {symbol}")
        print(f"{'='*60}")
        
        for model_name, model_data in data['results'].items():
            test_metrics = model_data['test_metrics']
            cm = test_metrics['confusion_matrix']
            threshold = model_data['best_threshold']
            
            print(f"\n{model_name} (阈值={threshold:.3f}):")
            print("-" * 40)
            
            # Print the confusion matrix.
            if cm.size == 4:
                tn, fp, fn, tp = cm.ravel()
                print(f"混淆矩阵:")
                print(f"           预测为负    预测为正")
                print(f"实际为负:  TN={tn:4d}     FP={fp:4d}")
                print(f"实际为正:  FN={fn:4d}     TP={tp:4d}")
                
                # Calculate percentages.
                total = tp + fp + fn + tn
                if total > 0:
                    print(f"\n百分比形式:")
                    print(f"           预测为负    预测为正")
                    print(f"实际为负:  {tn/total:6.2%}     {fp/total:6.2%}")
                    print(f"实际为正:  {fn/total:6.2%}     {tp/total:6.2%}")
                
                # Calculate key ratios.
                print(f"\n关键比率:")
                print(f"准确率: {test_metrics['accuracy']:.4f}")
                print(f"精确率: {test_metrics['precision']:.4f}")
                print(f"召回率: {test_metrics['recall']:.4f}")
                print(f"F1分数: {test_metrics['f1']:.4f}")
                
                # Calculate the fraction predicted to rise.
                pred_up_ratio = (tp + fp) / total
                actual_up_ratio = (tp + fn) / total
                print(f"预测为上涨的比例: {pred_up_ratio:.4f}")
                print(f"实际为上涨的比例: {actual_up_ratio:.4f}")
                
                # Check for the always-predict-up problem.
                if pred_up_ratio > 0.99:  # More than 99% predicted up
                    print(f"⚠️ 警告: 模型几乎总是预测上涨")
                elif pred_up_ratio < 0.01:  # More than 99% predicted down
                    print(f"⚠️ 警告: 模型几乎总是预测下跌")
                
                # Check whether performance matches an always-predict-up strategy.
                always_up_accuracy = actual_up_ratio
                always_up_precision = actual_up_ratio
                always_up_f1 = 2 * actual_up_ratio / (1 + actual_up_ratio) if actual_up_ratio > 0 else 0
                
                accuracy_diff = test_metrics['accuracy'] - always_up_accuracy
                f1_diff = test_metrics['f1'] - always_up_f1
                
                if abs(accuracy_diff) < 0.01 and abs(f1_diff) < 0.01:
                    print(f"⚠️ 模型表现与'总是预测上涨'策略几乎相同!")
                    
            else:
                print(f"混淆矩阵: {cm}")
                
    print(f"\n{'='*80}")
    print("混淆矩阵分析完成")
    print("="*80)
    
    # Plot confusion-matrix heatmaps.
    print("\n创建混淆矩阵热图...")
    
    fig, axes = plt.subplots(len(STOCK_SYMBOLS), 3, figsize=(15, 4*len(STOCK_SYMBOLS)))
    
    for i, symbol in enumerate(STOCK_SYMBOLS):
        if symbol in all_results:
            for j, model_name in enumerate(['TCN', 'DLinear', 'LSTM']):
                if model_name in all_results[symbol]['results']:
                    test_metrics = all_results[symbol]['results'][model_name]['test_metrics']
                    cm = test_metrics['confusion_matrix']
                    threshold = all_results[symbol]['results'][model_name]['best_threshold']
                    
                    ax = axes[i, j] if len(STOCK_SYMBOLS) > 1 else axes[j]
                    
                    if cm.size == 4:
                        tn, fp, fn, tp = cm.ravel()
                        cm_display = np.array([[tn, fp], [fn, tp]])
                        
                        # Create a heatmap.
                        sns.heatmap(cm_display, annot=True, fmt='d', cmap='Blues', 
                                   cbar=False, ax=ax, square=True)
                        
                        # Set labels.
                        ax.set_title(f'{symbol} - {model_name}\n(阈值={threshold:.3f})')
                        ax.set_xlabel('预测标签')
                        ax.set_ylabel('真实标签')
                        ax.set_xticklabels(['下跌', '上涨'])
                        ax.set_yticklabels(['下跌', '上涨'])
                        
                        # Add percentage annotations.
                        total = tp + fp + fn + tn
                        if total > 0:
                            for m in range(2):
                                for n in range(2):
                                    value = cm_display[m, n]
                                    percentage = value / total * 100
                                    ax.text(n + 0.5, m + 0.7, f'{percentage:.1f}%', 
                                           ha='center', va='center', fontsize=9, color='red')
                    else:
                        ax.text(0.5, 0.5, '无数据', ha='center', va='center')
                        ax.set_title(f'{symbol} - {model_name}')
                        
                    ax.grid(False)
    
    plt.tight_layout()
    plt.savefig('confusion_matrices.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ 混淆矩阵热图已保存为: confusion_matrices.png")
    
    # Create a summary table.
    print("\n混淆矩阵汇总表格:")
    print("="*80)
    
    summary_data_cm = []
    
    for symbol, data in all_results.items():
        for model_name, model_data in data['results'].items():
            test_metrics = model_data['test_metrics']
            cm = test_metrics['confusion_matrix']
            threshold = model_data['best_threshold']
            
            if cm.size == 4:
                tn, fp, fn, tp = cm.ravel()
                total = tp + fp + fn + tn
                pred_up_ratio = (tp + fp) / total if total > 0 else 0
                actual_up_ratio = (tp + fn) / total if total > 0 else 0
                
                # Check for the always-predict-up problem.
                always_up_warning = "⚠️" if pred_up_ratio > 0.99 else ""
                
                summary_data_cm.append({
                    'Stock': symbol,
                    'Model': model_name,
                    'Threshold': threshold,
                    'TP': tp,
                    'FP': fp,
                    'FN': fn,
                    'TN': tn,
                    'Total': total,
                    'Pred_Up_Ratio': pred_up_ratio,
                    'Actual_Up_Ratio': actual_up_ratio,
                    'Always_Up_Warning': always_up_warning
                })
    
    summary_cm_df = pd.DataFrame(summary_data_cm)
    print(summary_cm_df.to_string(index=False))
    
    # Save the summary table.
    summary_cm_df.to_csv('confusion_matrices_summary.csv', index=False)
    print(f"\n✓ 混淆矩阵汇总表格已保存为: confusion_matrices_summary.csv")
    
    # Analyze possible always-predict-up behavior.
    print("\n" + "="*80)
    print("是否存在'总是预测上涨'问题分析:")
    print("="*80)
    
    high_pred_up_count = 0
    for symbol, data in all_results.items():
        print(f"\n{symbol}:")
        for model_name, model_data in data['results'].items():
            test_metrics = model_data['test_metrics']
            cm = test_metrics['confusion_matrix']
            
            if cm.size == 4:
                tn, fp, fn, tp = cm.ravel()
                total = tp + fp + fn + tn
                pred_up_ratio = (tp + fp) / total if total > 0 else 0
                
                status = "⚠️ 总是预测上涨" if pred_up_ratio > 0.99 else "正常"
                if pred_up_ratio > 0.99:
                    high_pred_up_count += 1
                    
                print(f"  {model_name}: 预测上涨比例={pred_up_ratio:.4f} ({status})")
    
    print(f"\n总结:")
    print(f"  总模型数: {len(summary_cm_df)}")
    print(f"  可能'总是预测上涨'的模型数: {high_pred_up_count}")
    
    if high_pred_up_count / len(summary_cm_df) > 0.5:
        print(f"⚠️ 警告: 超过一半的模型可能存在问题")
    else:
        print(f"✓ 大部分模型似乎没有'总是预测上涨'的问题")
    
    # Compare with the 5-minute results.
    print("\n" + "="*80)
    print("与5分钟数据结果对比:")
    print("="*80)
    
    print("5分钟数据:")
    print("  - 所有模型在测试集上预测上涨比例=1.0 (总是预测上涨)")
    print("  - 最优阈值均为0.1")
    print("  - 模型表现与'总是预测上涨'策略完全相同")
    
    print("\n日线数据:")
    print("  - 预测上涨比例分布在0.4-1.0之间")
    print("  - 最优阈值分布在0.1-0.45之间")
    print("  - 不同股票、不同模型表现各异")
    
    print("\n结论:")
    print("  - 5分钟数据: 模型确实存在'总是预测上涨'的问题")
    print("  - 日线数据: 模型表现更正常，没有系统性地'总是预测上涨'")
    
else:
    print("❌ 无可用结果进行混淆矩阵分析")